# 🤖 Assignment 1 — Option 2: Chatbot with Conversational Memory

> **Cohort:** AI Engineer Ready — Cohort 1  
> **Difficulty:** ⭐⭐⭐ Intermediate  

---

## 📌 What This Assignment Is About

Most chatbots have **no memory**. Ask "What sport did I say I love?" and a forgetful bot will have no idea — it treats every message as brand new.

In this assignment, you will build a chatbot that:
- **Remembers** everything discussed in the current conversation
- **Generates context-aware replies** that reference what you said earlier
- **Persists memory to disk** (`memory.json`) so conversations survive script restarts

You will use **LangChain's `PromptTemplate`** and an **LLM** (OpenAI / Groq / Gemini) to power the responses.

---

## 📁 Files in This Assignment

| File | What It Does |
|---|---|
| `chatbot.py` | The main script — runs the chat loop, loads/saves memory. **You implement `generate_response()` here.** |
| `Chatbot_With_Memory.ipynb` | 👈 **This notebook** — build & test step by step |
| `requirements.txt` | Python packages to install |
| `memory.json` | Auto-created at runtime — stores conversation history |

---

## 🗺️ Your Roadmap

Work through this notebook **top to bottom**. Each section has:
- 📖 An explanation of what you're doing and why
- 🧱 Starter / scaffold code
- ✏️ A `# TODO` cell where you write your code
- ✅ A test cell to verify your work

**Once everything works here → copy `generate_response()` into `chatbot.py` → run end-to-end.**

---
## 🔧 Section 1: Environment Setup

Before writing any code, make sure your environment is ready.

### What you'll do here:
1. Install required packages
2. Load your API key from a `.env` file
3. Confirm the LLM can be initialised

### 📋 Evaluation Note:
> This section is not graded, but if your environment isn't set up correctly, **nothing else will work**. Take 5 minutes to get this right.

In [ ]:
# Run this cell once to install all required packages
# If you're on a virtual environment, activate it first

%pip install openai langchain langchain-openai langchain-core python-dotenv streamlit --quiet

In [ ]:
# Load your API key from a .env file in this folder
# Your .env file should contain:  OPENAI_API_KEY=sk-...

from dotenv import load_dotenv
import os

load_dotenv()  # Reads the .env file and sets environment variables

api_key = os.getenv("OPENAI_API_KEY")

if api_key:
    print("✅ API key loaded successfully:", api_key[:8], "...")
else:
    print("❌ API key not found. Create a .env file with OPENAI_API_KEY=your_key_here")

In [ ]:
# Quick sanity check — can we talk to the LLM?
# This should print a short greeting from the model.

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0.7)
response = llm.invoke("Say hello in one sentence.")
print(response.content)

---
## 📦 Section 2: Understanding the Memory Format

Before building the chatbot, you need to understand **how memory is stored and structured**.

The `memory` variable is a **Python list of dictionaries**. Each dictionary represents one message in the conversation.

```python
[
  {"role": "user",      "text": "Hi! My name is Priya."},
  {"role": "assistant", "text": "Hello Priya! How can I help you today?"},
  {"role": "user",      "text": "I want to learn about neural networks."}
]
```

This list is:
- **Loaded from `memory.json`** at the start of each session (so the bot remembers previous chats)
- **Appended to** after every user message and bot reply
- **Saved back to `memory.json`** after each exchange

### Your job in `generate_response()`:
Convert this list into a readable string, inject it into a prompt, and call the LLM.

### 📋 Evaluation Note:
> **30% of your grade** depends on correctly passing the history into the prompt and having it influence the bot's replies. Run the test cell below to make sure you understand the format before moving on.

In [ ]:
# Let's explore the memory format with a sample conversation

sample_memory = [
    {"role": "user",      "text": "Hi! My name is Priya."},
    {"role": "assistant", "text": "Hello Priya! How can I help you today?"},
    {"role": "user",      "text": "I want to learn about neural networks."}
]

# Print the raw memory list
print("Raw memory list:")
for i, msg in enumerate(sample_memory):
    print(f"  [{i}] role={msg['role']}, text={msg['text']}")

In [ ]:
# ✏️ TODO: Write a function that converts the memory list into a formatted string
# This string will be injected into the prompt as 'conversation history'
#
# Expected output for sample_memory above:
#   User: Hi! My name is Priya.
#   Assistant: Hello Priya! How can I help you today?
#   User: I want to learn about neural networks.
#
# Hints:
#   - Loop through each message dict in the memory list
#   - Use msg['role'].capitalize() to get 'User' or 'Assistant'
#   - Join the lines with '\n'

def format_memory(memory: list) -> str:
    # TODO: Replace this with your implementation
    pass


# Test it
formatted = format_memory(sample_memory)
print(formatted)

In [ ]:
# ✅ Verification cell — run this after completing the TODO above
# All assertions should pass with no errors

assert formatted is not None, "format_memory() returned None — did you forget to return?"
assert "Priya" in formatted, "The user message about Priya should appear in the formatted string"
assert "User:" in formatted, "Lines should start with 'User:'"
assert "Assistant:" in formatted, "Lines should start with 'Assistant:'"

print("✅ format_memory() looks correct!")

---
## 🗃️ Section 3: Load & Save Memory (JSON)

The bot needs to **remember conversations across multiple runs**. We do this by saving the memory list to a JSON file (`memory.json`) and loading it at startup.

The functions `load_memory()` and `save_memory()` are already written in `chatbot.py`. Here you'll understand and test them.

### Why this matters:
Without persistence, closing the terminal clears all memory. With `memory.json`, the bot picks up right where it left off.

### 📋 Evaluation Note:
> **15% of your grade** is for correctly saving and loading `memory.json`. The tests below verify your implementation.

In [ ]:
# These are the exact functions from chatbot.py — study them carefully

import json
import os

MEMORY_FILE = "memory.json"

def load_memory():
    """Load conversation history from memory.json. Returns empty list if file doesn't exist."""
    if os.path.exists(MEMORY_FILE):
        with open(MEMORY_FILE, "r", encoding="utf-8") as f:
            return json.load(f)
    return []

def save_memory(memory):
    """Save conversation history to memory.json."""
    with open(MEMORY_FILE, "w", encoding="utf-8") as f:
        json.dump(memory, f, indent=2)

print("Functions defined. Run the test cell below.")

In [ ]:
# ✅ Test load/save — saves a dummy conversation, reloads it, checks it matches

test_memory = [
    {"role": "user", "text": "Test message"},
    {"role": "assistant", "text": "Test reply"}
]

save_memory(test_memory)
loaded = load_memory()

assert loaded == test_memory, "Loaded memory doesn't match what was saved!"
print("✅ save_memory() and load_memory() work correctly!")
print("Contents of memory.json:", loaded)

---
## 🧠 Section 4: Building the Prompt with PromptTemplate

This is the **heart of the assignment**. You will:
1. Use LangChain's `PromptTemplate` to create a reusable prompt blueprint
2. Inject the conversation history (`{history}`) and latest user message (`{user_input}`) into it
3. Print the formatted prompt so you can see exactly what gets sent to the LLM

### What is a PromptTemplate?
Think of it like an **f-string on steroids**. You define a template with `{placeholders}` and fill them in at call time.

```python
from langchain_core.prompts import PromptTemplate

template = PromptTemplate(
    input_variables=["history", "user_input"],
    template="""..."""
)

filled = template.format(history="...", user_input="...")
```

### 📋 Evaluation Note:
> **30% of your grade** is for correctly using `PromptTemplate` with `{history}` and `{user_input}` variables. The prompt must include a system instruction telling the bot to remember context.

In [ ]:
# ✏️ TODO: Create a PromptTemplate with {history} and {user_input} placeholders
#
# Your template should contain:
#   1. A system instruction (e.g., "You are a helpful assistant. Use the conversation history to answer.")
#   2. A {history} section showing the past conversation
#   3. The current user message: {user_input}
#   4. A cue for the assistant to reply (e.g., "Assistant:")
#
# Example structure:
#   You are a helpful assistant with memory...
#
#   Conversation so far:
#   {history}
#
#   User: {user_input}
#   Assistant:

from langchain_core.prompts import PromptTemplate

prompt_template = PromptTemplate(
    input_variables=["history", "user_input"],
    template="""TODO: Write your prompt template here"""
)

# Test: format the prompt with sample data
sample_history = "User: Hi! My name is Priya.\nAssistant: Hello Priya!"
filled_prompt = prompt_template.format(
    history=sample_history,
    user_input="What is my name?"
)
print("=== Formatted Prompt ===")
print(filled_prompt)

In [ ]:
# ✅ Verification cell

assert "{history}" not in filled_prompt, "{history} placeholder was not replaced!"
assert "{user_input}" not in filled_prompt, "{user_input} placeholder was not replaced!"
assert "Priya" in filled_prompt, "History content should appear in the formatted prompt"
assert "What is my name?" in filled_prompt, "User input should appear in the formatted prompt"

print("✅ PromptTemplate is set up correctly!")

---
## ⛓️ Section 5: Building the LCEL Chain (Prompt → LLM)

Now that your prompt is ready, you'll connect it to the LLM using **LangChain's LCEL** (LangChain Expression Language).

### What is LCEL?
LCEL uses the `|` (pipe) operator to chain components together — just like Unix pipes.

```python
chain = prompt_template | llm
result = chain.invoke({"history": "...", "user_input": "..."})
```

This is the **modern LangChain approach** (preferred over the older `LLMChain`).

### What `.invoke()` returns:
A `AIMessage` object. To get the text: `result.content`

### 📋 Evaluation Note:
> Using LCEL (`|` pipe syntax) and `.invoke()` is required. Old-style `LLMChain` is accepted only if documented.

In [ ]:
# ✏️ TODO: Create an LCEL chain by piping prompt_template into your LLM
#
# Steps:
#   1. Instantiate ChatOpenAI with temperature=0.7
#   2. Create the chain:  chain = prompt_template | llm
#   3. Call chain.invoke() with a dict of {"history": ..., "user_input": ...}
#   4. Extract and print the reply text from result.content
#
# Use the sample_history variable from the previous section as test input.

from langchain_openai import ChatOpenAI

# TODO: Initialise the LLM
llm = None  # Replace with ChatOpenAI(...)

# TODO: Create the LCEL chain
chain = None  # Replace with prompt_template | llm

# TODO: Call chain.invoke() with sample data
result = None  # Replace with chain.invoke({...})

# TODO: Print the reply
print(result)  # Should be a string like "Your name is Priya."

---
## 🔨 Section 6: Implementing `generate_response()`

Now you'll bring everything together into the **single function** that powers the chatbot.

### Function signature:
```python
def generate_response(user_input: str, memory: list) -> str:
```

### What it must do (checklist):
- [ ] Call `format_memory(memory)` to convert the list into a string
- [ ] Use `prompt_template` with `{history}` and `{user_input}` placeholders
- [ ] Call the LCEL chain with `.invoke()`
- [ ] Return the reply as a **plain string** (not an `AIMessage` object)

### 📋 Evaluation Note:
> This single function is worth **30% of your grade** directly, plus it underpins the 30% for memory and the 10% for end-to-end execution. Get this right before moving to Section 7.

In [ ]:
# ✏️ TODO: Implement generate_response() using everything from Sections 2–5
#
# Do NOT use hardcoded responses.
# Do NOT skip the memory — the history MUST influence the reply.
#
# Reminder of format_memory() output:
#   "User: Hi! My name is Priya.\nAssistant: Hello Priya!"
#
# Reminder of chain.invoke() input:
#   chain.invoke({"history": history_str, "user_input": user_input})
#
# Return result.content (a plain string), not the full AIMessage object.

def generate_response(user_input: str, memory: list) -> str:
    """
    Takes the user's latest message and full conversation history.
    Returns the bot's reply as a plain string.
    """
    # TODO: Step 1 — Format the memory list into a string using format_memory()
    history_str = None

    # TODO: Step 2 — Invoke the LCEL chain with history_str and user_input
    result = None

    # TODO: Step 3 — Return the reply as a plain string
    return None


print("generate_response() defined. Run the test cell below.")

In [ ]:
# ✅ Test generate_response() with a memory-dependent question
# The bot should answer 'cricket' without it being mentioned in user_input

test_mem = [
    {"role": "user",      "text": "My name is Arjun and I love cricket."},
    {"role": "assistant", "text": "Nice to meet you Arjun! Cricket is a great sport."}
]

reply = generate_response("What sport did I say I love?", test_mem)
print("Bot:", reply)

assert isinstance(reply, str), "generate_response() must return a string"
assert len(reply) > 5, "Reply is too short — something may be wrong"
assert reply != "This is a placeholder response. Replace with LLM call.", "You forgot to replace the placeholder!"

# Soft check — the bot should reference cricket
if "cricket" in reply.lower():
    print("✅ Bot correctly remembered cricket from the conversation history!")
else:
    print("⚠️  Bot did not mention cricket — check that history is being passed into the prompt correctly.")

---
## 🔁 Section 7: Simulating the Full Chat Loop

Now let's simulate the actual conversation loop that `chatbot.py` runs — but inside the notebook so you can test without switching to the terminal.

This is the same logic as `main()` in `chatbot.py`:
1. Load memory from file
2. Accept user input
3. Append to memory
4. Call `generate_response()`
5. Append reply to memory
6. Save memory to file

### 📋 Evaluation Note:
> **10% of your grade** is for the full end-to-end loop running without errors. This section validates exactly that.

In [ ]:
# Simulate a 3-turn conversation programmatically (no manual typing)
# This mirrors the logic inside main() in chatbot.py

# Start fresh for this test
simulated_memory = []

test_inputs = [
    "My name is Arjun and I love cricket.",
    "What sport did I say I love?",
    "Suggest a Python project related to what I enjoy."
]

for user_msg in test_inputs:
    print(f"You: {user_msg}")
    simulated_memory.append({"role": "user", "text": user_msg})
    
    reply = generate_response(user_msg, simulated_memory)
    print(f"Bot: {reply}")
    print("-" * 60)
    
    simulated_memory.append({"role": "assistant", "text": reply})

# Save to memory.json so you can inspect it
save_memory(simulated_memory)
print("\n✅ Conversation saved to memory.json")

In [ ]:
# ✅ Verify that memory.json was written correctly

reloaded = load_memory()
assert len(reloaded) == 6, f"Expected 6 messages (3 turns × 2), got {len(reloaded)}"
assert reloaded[0]["role"] == "user", "First message should be from user"
assert reloaded[1]["role"] == "assistant", "Second message should be from assistant"

print("✅ memory.json persisted correctly with", len(reloaded), "messages.")

---
## 🧹 Section 8: Code Quality & Comments

Clean, readable, well-commented code is part of your grade.

### Checklist before submitting:
- [ ] All `# TODO` cells are replaced with real code
- [ ] Your functions have docstrings explaining what they do
- [ ] Variable names are descriptive (not `x`, `temp`, `data`)
- [ ] No unused imports or dead code
- [ ] Inline comments explain *why*, not just *what*

### 📋 Evaluation Note:
> **15% of your grade** is code quality. The cell below is a self-checklist — fill it in honestly before submitting.

In [ ]:
# ✏️ Self-assessment — change False to True for each item you've completed

checklist = {
    "All TODO cells replaced with real code":           False,  # TODO: Change to True
    "generate_response() has a docstring":              False,  # TODO: Change to True
    "format_memory() has a docstring":                  False,  # TODO: Change to True
    "No hardcoded placeholder response remains":        False,  # TODO: Change to True
    "Inline comments explain key steps":                False,  # TODO: Change to True
    "Implementation copied into chatbot.py":            False,  # TODO: Change to True
    "chatbot.py runs end-to-end without errors":        False,  # TODO: Change to True
}

completed = sum(checklist.values())
total = len(checklist)

print(f"\nCode Quality Checklist: {completed}/{total} completed\n")
for item, done in checklist.items():
    status = "✅" if done else "❌"
    print(f"  {status}  {item}")

---
## ✅ Section 9: Final Submission Checklist

Before you submit, go through this list:

### Core Requirements
- [ ] `format_memory()` converts the memory list to a readable string ✔
- [ ] `PromptTemplate` uses `{history}` and `{user_input}` variables ✔
- [ ] LCEL chain created using `prompt_template | llm` ✔
- [ ] `generate_response()` returns a real LLM reply (not the placeholder) ✔
- [ ] Memory influences the bot's answers (cricket test passes) ✔
- [ ] `memory.json` is saved and loaded correctly ✔
- [ ] Implementation is copied into `chatbot.py` and runs end-to-end ✔

### Code Quality
- [ ] Functions have docstrings
- [ ] Inline comments explain key decisions
- [ ] No dead code or unused imports

### Files to Submit
- `chatbot.py` — with your `generate_response()` implementation
- `Chatbot_With_Memory.ipynb` — this notebook, fully run (all cells have output)
- `memory.json` — the file generated by your final test run
- `requirements.txt` — unchanged

---

## 📊 Grading Breakdown

| Criteria | Weight |
|---|---|
| `generate_response()` uses PromptTemplate + LLM (not hardcoded) | 30% |
| Conversation history is passed into the prompt and influences replies | 30% |
| `memory.json` is correctly saved and loaded between runs | 15% |
| Code is clean, readable, and well-commented | 15% |
| Runs end-to-end via `chatbot.py` without errors | 10% |

---

> 🧠 **Pro Tip:** The quality of your chatbot depends almost entirely on how well you construct the `PromptTemplate`. Spend real time on it — inject the history cleanly, write a clear system instruction, and iterate. That's the core skill being assessed.

*Happy building! The scaffolding is ready — your job is to bring it to life. 🚀*